In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import math
from itertools import permutations

import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib import colors
from matplotlib import gridspec
from matplotlib.colors import Normalize, TwoSlopeNorm
import cartopy.feature as cfeature
import cartopy.crs as ccrs
import seaborn as sns

In [2]:
ds = xr.open_dataset("/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/bootstrap_summary.nc",
                    engine='h5netcdf')

In [3]:
lon = ds.coords['lon'].values
lat = ds.coords['lat'].values

In [4]:
ds

<xarray.Dataset> Size: 96MB
Dimensions:                 (hour: 24, lon: 125, lat: 163)
Coordinates:
  * hour                    (hour) int8 24B 0 1 2 3 4 5 6 ... 18 19 20 21 22 23
    height                  float64 8B ...
  * lon                     (lon) float64 1kB 147.0 147.1 147.1 ... 151.9 152.0
  * lat                     (lat) float64 1kB -38.49 -38.45 ... -32.05 -32.01
    crs                     int32 4B ...
Data variables: (12/52)
    hw_u_mean               (lat, lon, hour) float32 2MB ...
    hw_u_std                (lat, lon, hour) float32 2MB ...
    hw_u_ci_low             (lat, lon, hour) float32 2MB ...
    hw_u_ci_high            (lat, lon, hour) float32 2MB ...
    hw_v_mean               (lat, lon, hour) float32 2MB ...
    hw_v_std                (lat, lon, hour) float32 2MB ...
    ...                      ...
    diff_magnitude_pval     (lat, lon, hour) float32 2MB ...
    diff_joint_pval         (lat, lon, hour) float32 2MB ...
    diff_sig_magnitude_005  (lat, lon, hour) bool 489kB ...
    diff_sig_magnitude_001  (lat, lon, hour) bool 489kB ...
    diff_sig_joint_005      (lat, lon, hour) bool 489kB ...
    diff_sig_joint_001      (lat, lon, hour) bool 489kB ...

In [18]:
def simple_plot(lon, lat, 
                scalar=None,           # 2D array for background color
                vector=None,           # tuple (u, v) for wind vectors
                stipple=None,          # 2D boolean array for significance dots
                cmap="viridis",
                norm=None,             # matplotlib normalization (e.g., TwoSlopeNorm for diverging)
                scalar_label="",
                vector_color="black",
                vector_alpha=0.85,
                vector_scale=50,
                vector_subsample=6,
                stipple_size=0.1,
                stipple_color="black",
                title="",
                figsize=(10, 6)):
    """
    Simple plotting function for meteorological data.
    
    Parameters:
    -----------
    lon, lat : 1D arrays
        Longitude and latitude coordinates
    scalar : 2D array, optional
        Background field to color (e.g., wind speed, temperature)
    vector : tuple of 2D arrays (u, v), optional
        Wind vector components
    stipple : 2D boolean array, optional
        Significance mask for stippling
    norm : matplotlib.colors.Normalize, optional
        Normalization for colormap (e.g., TwoSlopeNorm for diverging colormaps)
    """
    
    # Check for cartopy
    have_cartopy = False
    try:
        import cartopy.crs as ccrs
        have_cartopy = True
        projection = ccrs.PlateCarree()
        data_crs = ccrs.PlateCarree()
    except ImportError:
        pass
    
    # Create figure
    if have_cartopy:
        fig, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": projection})
    else:
        fig, ax = plt.subplots(figsize=figsize)
    
    # Plot scalar background
    if scalar is not None:
        if have_cartopy:
            mesh = ax.pcolormesh(lon, lat, scalar, transform=data_crs, cmap=cmap, norm=norm)
        else:
            mesh = ax.pcolormesh(lon, lat, scalar, shading="auto", cmap=cmap, norm=norm)
        plt.colorbar(mesh, ax=ax, pad=0.02, label=scalar_label)
    
    # Plot vectors
    if vector is not None:
        u, v = vector
        sl_lat = slice(0, None, vector_subsample)
        sl_lon = slice(0, None, vector_subsample)
        
        if have_cartopy:
            ax.quiver(lon[sl_lon], lat[sl_lat],
                     u[sl_lat, sl_lon], v[sl_lat, sl_lon],
                     transform=data_crs,
                     color=vector_color, alpha=vector_alpha,
                     scale=vector_scale, width=0.0022, zorder=3)
        else:
            lon2d, lat2d = np.meshgrid(lon, lat)
            ax.quiver(lon2d[sl_lat, sl_lon], lat2d[sl_lat, sl_lon],
                     u[sl_lat, sl_lon], v[sl_lat, sl_lon],
                     color=vector_color, alpha=vector_alpha,
                     scale=vector_scale, width=0.0022, zorder=3)
    
    # Plot stippling
    if stipple is not None:
        yy, xx = np.where(stipple)
        if len(yy):
            if have_cartopy:
                ax.scatter(lon[xx], lat[yy],
                          s=stipple_size, c=stipple_color,
                          marker="o", alpha=0.9,
                          transform=data_crs, zorder=5)
            else:
                ax.scatter(lon[xx], lat[yy],
                          s=stipple_size, c=stipple_color,
                          marker="o", alpha=0.9, zorder=5)
    
    # Add coastlines if cartopy available
    if have_cartopy:
        ax.coastlines(linewidth=0.8, zorder=6)
    else:
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
    
    # Set title
    if title:
        ax.set_title(title)
    
    # Add legend if needed
    legend_items = []
    if vector is not None:
        from matplotlib.lines import Line2D
        legend_items.append(Line2D([0],[0], color=vector_color, lw=1.5, label="Vector"))
    if stipple is not None:
        from matplotlib.lines import Line2D
        legend_items.append(Line2D([0],[0], marker="o", linestyle="none",
                                  markerfacecolor=stipple_color,
                                  markersize=4, label="Significant"))
    
    if legend_items:
        ax.legend(handles=legend_items, loc="upper right", frameon=True)
    
    plt.tight_layout()
    return fig, ax

In [19]:
hour=13
u_wind = ds['diff_u_mean'].sel(hour=hour)
v_wind = ds['diff_v_mean'].sel(hour=hour)
wind_speed = ds['speed_difference_mean'].sel(hour=hour)
sig = ds['diff_joint_pval'].sel(hour=hour)

In [20]:
# Example usage:
scale = max([abs(wind_speed.min()),abs(wind_speed.max())])
norm = colors.TwoSlopeNorm(vmin=-scale, vcenter=0, vmax=scale)

# fig, ax = simple_plot(lon, lat, 
#                       scalar=wind_speed, 
#                       vector=(u_wind, v_wind),
#                       stipple=sig,
#                       cmap='coolwarm',
#                       norm=norm,
#                       scalar_label="Wind Speed (m/s)",
#                       title="Wind Analysis")

In [27]:
def facet_plot_hours(
        lon, lat, scalar, u=None, v=None, stipple=None,
        hours=None, ncols=4, figsize=(14, 8),
        cmap='viridis', norm=None,
        vmin=None, vmax=None,
        center_diff=False, center_value=0.0, symmetric_if_center=True,
        vector_scale=50, vector_subsample=(6, 6),
        stipple_size=6, stipple_color='k',
        title_prefix='',
        quiver_color='white', quiver_width=0.003,
        share_colorbar=True, colorbar_label='',
        add_borders=True, add_states=True,
        title=None,
        return_norm=False,  # NEW: return the norm used
        return_mappable=False  # NEW: return the last pcolormesh mappable
    ):
    """
    Faceted plot by hour using Matplotlib + Cartopy, with state and coastal borders.

    Parameters
    ----------
    lon, lat : 1D arrays
        Longitude and latitude coordinates.
    scalar : ndarray
        3D array with dimensions containing (hour, lat, lon) in some order. Will be
        auto-permuted to (hour, lat, lon).
    u, v : ndarray, optional
        3D wind component arrays on same grid/order as scalar. Will be permuted similarly.
    stipple : ndarray(bool or float), optional
        3D array. If float, values < 0.05 (default logic) are treated as True for stippling.
    hours : list[int], optional
        Subset of hour indices to plot.
    vmin, vmax : float, optional
        Explicit color scale limits (ignored if norm is provided).
    norm : matplotlib.colors.Normalize or subclass, optional
        If provided, used directly. Overrides vmin/vmax logic.
    center_diff : bool
        If True, use a TwoSlopeNorm centered at center_value.
        Priority order:
          1. If norm is provided: use it
          2. Else if center_diff True:
               a. If vmin/vmax given and symmetric_if_center True -> expand to symmetric about center_value
               b. Else if vmin/vmax given -> honor them literally
               c. Else auto-compute symmetric bounds from data
    center_value : float
        Center for TwoSlopeNorm if center_diff is True.
    symmetric_if_center : bool
        Whether to force symmetry around center_value when center_diff=True and user supplies vmin/vmax.
    share_colorbar : bool
        Draw a colorbar in an extra column.
    return_norm : bool
        If True, include the norm used in the return tuple.
    return_mappable : bool
        If True, include the last pcolormesh (for external colorbar creation).

    Returns
    -------
    fig : Figure
    axes_flat : list[Axes]
    (optional) norm_used
    (optional) last_mappable

    Notes
    -----
    To ensure multiple calls share identical scaling, build a Normalize (or TwoSlopeNorm)
    externally and pass via norm. Example:

        shared_norm = Normalize(vmin=-5, vmax=15)
        fig1, ax1 = facet_plot_hours(..., norm=shared_norm, share_colorbar=False)
        fig2, ax2 = facet_plot_hours(..., norm=shared_norm, share_colorbar=False)
        # Then create a single colorbar:
        import matplotlib.pyplot as plt
        cbar = plt.colorbar(plt.cm.ScalarMappable(norm=shared_norm, cmap='viridis'),
                            ax=ax1 + ax2, orientation='horizontal')
    """
    # --- Ensure numpy arrays ---
    scalar = np.asarray(scalar)
    if u is not None: u = np.asarray(u)
    if v is not None: v = np.asarray(v)
    if stipple is not None: stipple = np.asarray(stipple)

    if scalar.ndim != 3:
        raise ValueError(f"Expect 3D scalar array, got shape {scalar.shape}")

    nh_guess, nlat_guess, nlon_guess = scalar.shape
    # --- Auto-reorder scalar to (hour, lat, lon) ---
    found = False
    for perm in permutations(range(3)):
        test = scalar.transpose(perm)
        if test.shape[1] == len(lat) and test.shape[2] == len(lon):
            scalar = test
            if u is not None: u = u.transpose(perm)
            if v is not None: v = v.transpose(perm)
            if stipple is not None: stipple = stipple.transpose(perm)
            found = True
            break
    if not found:
        raise ValueError(f"Cannot reconcile scalar shape {scalar.shape} with lat={len(lat)}, lon={len(lon)}")

    nh = scalar.shape[0]

    # --- Select hours ---
    if hours is None:
        sel_indices = list(range(nh))
        print(sel_indices)
        return sel_indices
    else:
        hour_coord = np.arange(nh)
        sel_indices = []
        for h in hours:
            if h in hour_coord:
                sel_indices.append(int(np.where(hour_coord == h)[0][0]))
            elif 0 <= h < nh:
                sel_indices.append(int(h))
            else:
                raise ValueError(f"Hour {h} not found")
    nplots = len(sel_indices)
    ncols = max(1, ncols)
    nrows = math.ceil(nplots / ncols)

    # --- Figure & GridSpec (reserve last col for colorbar if share_colorbar) ---
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(nrows, ncols + (1 if share_colorbar else 0),
                           figure=fig,
                           wspace=0.02, hspace=0.08,
                           width_ratios=[1]*ncols + ([0.2] if share_colorbar else []))
    axes_flat = []

    # --- Determine normalization ---
    panel_data = scalar[sel_indices]

    if norm is not None:
        norm_used = norm
    else:
        if center_diff:
            # Determine bounds
            if vmin is not None or vmax is not None:
                # If only one is provided, try to infer counterpart from data
                data_min = np.nanmin(panel_data)
                data_max = np.nanmax(panel_data)
                vmin_eff = vmin if vmin is not None else data_min
                vmax_eff = vmax if vmax is not None else data_max
                if symmetric_if_center:
                    # Make symmetric about center_value
                    dist_min = center_value - vmin_eff
                    dist_max = vmax_eff - center_value
                    bound = max(dist_min, dist_max)
                    vmin_eff = center_value - bound
                    vmax_eff = center_value + bound
                norm_used = TwoSlopeNorm(vcenter=center_value, vmin=vmin_eff, vmax=vmax_eff)
            else:
                dmin = np.nanmin(panel_data)
                dmax = np.nanmax(panel_data)
                # Symmetric about center_value
                dist_min = center_value - dmin
                dist_max = dmax - center_value
                bound = max(dist_min, dist_max)
                norm_used = TwoSlopeNorm(vcenter=center_value,
                                         vmin=center_value - bound,
                                         vmax=center_value + bound)
        else:
            # Simple Normalize
            if vmin is None: vmin = np.nanmin(panel_data)
            if vmax is None: vmax = np.nanmax(panel_data)
            norm_used = Normalize(vmin=vmin, vmax=vmax)

    # --- Vector subsampling prep ---
    if u is not None and v is not None:
        si, sj = vector_subsample
        lat_vec = lat[::si]
        lon_vec = lon[::sj]
        Lon2d_vec, Lat2d_vec = np.meshgrid(lon_vec, lat_vec)

    # --- Full meshgrid for stipple ---
    Lon2d, Lat2d = np.meshgrid(lon, lat)

    last_mappable = None

    # --- Plot each panel ---
    for i, idx in enumerate(sel_indices):
        row = i // ncols
        col = i % ncols
        ax = fig.add_subplot(gs[row, col], projection=ccrs.PlateCarree())
        axes_flat.append(ax)

        ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='lightgray', zorder=0)
        if add_borders:
            ax.add_feature(cfeature.BORDERS.with_scale('50m'), edgecolor='black', linewidth=0.5, zorder=5)
        if add_states:
            ax.add_feature(cfeature.STATES.with_scale('50m'), edgecolor='black', linewidth=0.5, zorder=6)

        panel = scalar[idx]
        pm = ax.pcolormesh(lon, lat, panel, cmap=cmap, norm=norm_used,
                           shading='auto', transform=ccrs.PlateCarree(), zorder=1)
        last_mappable = pm  # keep reference

        if u is not None and v is not None:
            si, sj = vector_subsample
            u_i = u[idx][::si, ::sj]
            v_i = v[idx][::si, ::sj]
            ax.quiver(Lon2d_vec, Lat2d_vec, u_i, v_i, angles='xy', scale=vector_scale,
                      color=quiver_color, width=quiver_width,
                      transform=ccrs.PlateCarree(), zorder=7)

        if stipple is not None:
            mask_i = stipple[idx]
            if mask_i.dtype != bool and np.issubdtype(mask_i.dtype, np.floating):
                mask_i = mask_i < 0.05
            if mask_i.shape != (len(lat), len(lon)):
                raise ValueError(f"Stipple shape {mask_i.shape} does not match lat/lon ({len(lat)}, {len(lon)})")
            yy, xx = np.where(mask_i)
            if yy.size > 0:
                ax.scatter(Lon2d[yy, xx], Lat2d[yy, xx],
                           s=stipple_size, c=stipple_color,
                           marker='.', linewidths=0,
                           transform=ccrs.PlateCarree(),
                           zorder=8)

        ax.set_title(f"Hour {int(idx):02d}", fontsize=10)
        ax.set_xticks([])
        ax.set_yticks([])

    # --- Hide unused axes if share_colorbar shrinks layout? Not needed unless we added extras ---
    total_slots = nrows * ncols
    if nplots < total_slots:
        for j in range(nplots, total_slots):
            row = j // ncols
            col = j % ncols
            ax = fig.add_subplot(gs[row, col])
            ax.set_visible(False)

    # --- Colorbar ---
    if share_colorbar and last_mappable is not None:
        cbar_ax = fig.add_subplot(gs[:, -1])
        cbar = fig.colorbar(last_mappable, cax=cbar_ax)
        if colorbar_label:
            cbar.set_label(colorbar_label)

    if title:
        fig.suptitle(title, fontsize=16, y=0.925)

    returns = [fig, axes_flat]
    if return_norm:
        returns.append(norm_used)
    if return_mappable:
        returns.append(last_mappable)
    if len(returns) == 2:
        return fig, axes_flat
    return tuple(returns)

In [28]:
fig, axs = facet_plot_hours(
    lon=ds['lon'].values,
    lat=ds['lat'].values,
    scalar=ds['hw_speed_mean'].values,
    u=ds['hw_u_mean'].values,
    v=ds['hw_v_mean'].values,
    hours=None,
    ncols=6,
    figsize=(20, 16),
    cmap='cividis',
    colorbar_label='Wind speed (m/s)',
    title = "Wind composite of heatwave days"
    )
plt.show()


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


ValueError: too many values to unpack (expected 2)

In [29]:
fig, axs = facet_plot_hours(
    lon=ds['lon'].values,
    lat=ds['lat'].values,
    scalar=ds['bl_speed_mean'].values,
    u=ds['bl_u_mean'].values,
    v=ds['bl_v_mean'].values,
    hours=None,
    ncols=6,
    figsize=(20, 16),
    cmap='cividis',
    colorbar_label='Wind speed (m/s)',
    title = "Wind composite of baseline days"
    )
plt.show()


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


ValueError: too many values to unpack (expected 2)

In [30]:
facet_plot_hours(
    lon=ds['lon'].values,
    lat=ds['lat'].values,
    scalar=ds['bl_speed_mean'].values,
    u=ds['bl_u_mean'].values,
    v=ds['bl_v_mean'].values,
    hours=None,
    ncols=6,
    figsize=(20, 16),
    cmap='cividis',
    colorbar_label='Wind speed (m/s)',
    title = "Wind composite of baseline days"
    )

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]


[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23]